# 그래프 신경망 실습

**Graph Neural Network · GNN**

노드와 연결로 표현된 데이터에서 관계를 학습하는 신경망.

소재 분야에서 이해하기: 원자를 노드, 이웃 관계를 연결로 표현해 물성을 예측한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [소재 발견용 그래프 신경망 연구](https://www.nature.com/articles/s41586-023-06735-9)

## 1. 메시지 패싱을 직접 구현

결정 구조를 흉내낸 그래프에서, 이웃 정보를 모아 노드 표현을 갱신하는 과정을 numpy로 짭니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def ring_graph(size, defect_at=None):
    """원자 size개가 고리로 연결된 구조. defect_at 위치의 원소만 다릅니다."""
    adjacency = np.zeros((size, size))
    for i in range(size):
        adjacency[i, (i + 1) % size] = adjacency[(i + 1) % size, i] = 1
    features = np.zeros((size, 2)); features[:, 0] = 1.0   # 원소 A
    if defect_at is not None:
        features[defect_at] = [0.0, 1.0]                   # 원소 B
    return adjacency, features

adjacency, features = ring_graph(8, defect_at=3)
print(adjacency.astype(int))
print(features)

In [ ]:
def message_passing(adjacency, features, rounds=3, seed=0):
    """이웃 특징의 평균을 모아 자기 특징과 섞습니다(가장 단순한 GNN 층)."""
    local = np.random.default_rng(seed)
    width = features.shape[1]
    W_self = local.normal(0, 0.6, (width, width))
    W_neighbour = local.normal(0, 0.6, (width, width))
    degree = adjacency.sum(1, keepdims=True)
    hidden = features
    for _ in range(rounds):
        aggregated = (adjacency @ hidden) / np.maximum(degree, 1)
        hidden = np.tanh(hidden @ W_self + aggregated @ W_neighbour)
    return hidden

hidden = message_passing(adjacency, features)
print('노드별 표현 (결함 원자 3번과 이웃 2·4번이 달라졌는지 확인)')
for index, row in enumerate(hidden):
    print('  원자 %d %s' % (index, np.round(row, 3)))
print('\n구조 전체 표현(노드 평균) =', np.round(hidden.mean(0), 3))

## 2. 서로 다른 구조는 다른 표현을 갖습니다

In [ ]:
for label, graph in [('결함 없음', ring_graph(8)), ('결함 1개', ring_graph(8, 3)),
                     ('결함 2개', ring_graph(8, [2, 5]))]:
    a, f = graph
    print('%-10s 구조 표현 %s' % (label, np.round(message_passing(a, f).mean(0), 3)))
print('\n실제 GNN은 이 표현을 물성 라벨에 맞게 학습합니다. 여기서는 가중치를 무작위로 두고 구조 차이만 확인했습니다.')

## 3. 해석

노드는 라운드를 돌 때마다 더 먼 이웃의 정보까지 담습니다. 결정 구조를 원자 그래프로 바꾸면
조성만으로는 구분되지 않는 구조 차이를 모델에 넣을 수 있습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#gnn)을 여세요.